#   Actividad 3 — Infraestructura Virtual con Databricks y Big Data

  **Asignatura:** Big Data
  **Estudiante:** Juan Felipe Valencia Renteria
  **Fecha:** 2026-06-07
  **Entorno:** Databricks Community Edition — Serverless Compute

  ---

  ## Objetivo

  Desplegar el dataset `users_connectatel.csv` sobre una infraestructura virtual en Databricks CE, diseñando el esquema
  de almacenamiento, configurando la arquitectura básica, cargando los datos e ingiriendo la tabla. Validando el
  procesamiento con Spark y SQL.

  ---

  ## Analisis de Viabilidad del Dataset

  El dataset `users_connectatel.csv` representa usuarios de una empresa de telecomunicaciones ficticia llamada
  **Connectatel**.

  | Criterio | Detalle | Viable |
  |---|---|---|
  | Volumen | 4.000 filas — suficiente para operaciones Spark | Si |
  | Tipos de datos | Enteros, strings, timestamps, decimales | Si |
  | Nulabilidad | `city` con valores ausentes (`?`), `churn_date` vacio para usuarios activos | Si |
  | Cardinalidad para GROUP BY | Campo `plan` (Basico / Premium), campo `city` (multiples ciudades) | Si |
  | Contexto de negocio | Analisis de churn en telecomunicaciones — con sentido analitico | Si |

  **Conclusion:** El dataset satisface todos los requerimientos de la actividad: permite diseñar un esquema con tipos
  variados, ejecutar validaciones con Spark y SQL, y realizar agrupaciones con significado de negocio.

  ---

##   Seccion 1 — Diseño del Esquema

###   1.1 Diccionario de Datos

  | # | Campo | Tipo PySpark | Nulable | Llave | Descripcion |
  |---|---|---|---|---|---|
  | 1 | user_id | IntegerType | No | PK | Identificador unico del usuario |
  | 2 | first_name | StringType | No | — | Nombre del usuario |
  | 3 | last_name | StringType | No | — | Apellido del usuario |
  | 4 | age | IntegerType | No | — | Edad del usuario en años |
  | 5 | city | StringType | Si | — | Ciudad de residencia (puede ser nulo o `?`) |
  | 6 | reg_date | TimestampType | No | — | Fecha y hora de registro en la plataforma |
  | 7 | plan | StringType | No | — | Plan contratado: `Basico` o `Premium` |
  | 8 | churn_date | TimestampType | Si | — | Fecha de baja del servicio (nulo si sigue activo) |
  | 9 | messages_included | IntegerType | No | — | Mensajes incluidos en el plan mensual |
  | 10 | gb_per_month | IntegerType | No | — | Gigabytes de datos incluidos por mes |
  | 11 | minutes_included | IntegerType | No | — | Minutos de llamada incluidos por mes |
  | 12 | usd_monthly_pay | DoubleType | No | — | Pago mensual en dolares |
  | 13 | usd_per_gb | DoubleType | No | — | Costo por GB adicional en dolares |
  | 14 | usd_per_message | DoubleType | No | — | Costo por mensaje adicional en dolares |
  | 15 | usd_per_minute | DoubleType | No | — | Costo por minuto adicional en dolares |

In [0]:
from pyspark.sql.types import (
      StructType, StructField,
      IntegerType, StringType, TimestampType, DoubleType
  )

schema_connectatel = StructType([
      StructField("user_id",           IntegerType(),   nullable=False),
      StructField("first_name",        StringType(),    nullable=False),
      StructField("last_name",         StringType(),    nullable=False),
      StructField("age",               IntegerType(),   nullable=False),
      StructField("city",              StringType(),    nullable=True),
      StructField("reg_date",          TimestampType(), nullable=False),
      StructField("plan",              StringType(),    nullable=False),
      StructField("churn_date",        TimestampType(), nullable=True),
      StructField("messages_included", IntegerType(),   nullable=False),
      StructField("gb_per_month",      IntegerType(),   nullable=False),
      StructField("minutes_included",  IntegerType(),   nullable=False),
      StructField("usd_monthly_pay",   DoubleType(),    nullable=False),
      StructField("usd_per_gb",        DoubleType(),    nullable=False),
      StructField("usd_per_message",   DoubleType(),    nullable=False),
      StructField("usd_per_minute",    DoubleType(),    nullable=False),
])

print("Esquema definido correctamente:")
for field in schema_connectatel.fields:
    nullable_str = "SI" if field.nullable else "NO"
    print(f"  {field.name:<25} {str(field.dataType):<20} Nulable: {nullable_str}")

Esquema definido correctamente:
  user_id                   IntegerType()        Nulable: NO
  first_name                StringType()         Nulable: NO
  last_name                 StringType()         Nulable: NO
  age                       IntegerType()        Nulable: NO
  city                      StringType()         Nulable: SI
  reg_date                  TimestampType()      Nulable: NO
  plan                      StringType()         Nulable: NO
  churn_date                TimestampType()      Nulable: SI
  messages_included         IntegerType()        Nulable: NO
  gb_per_month              IntegerType()        Nulable: NO
  minutes_included          IntegerType()        Nulable: NO
  usd_monthly_pay           DoubleType()         Nulable: NO
  usd_per_gb                DoubleType()         Nulable: NO
  usd_per_message           DoubleType()         Nulable: NO
  usd_per_minute            DoubleType()         Nulable: NO


In [0]:
%sql

  -- DDL equivalente en Spark SQL
  CREATE TABLE IF NOT EXISTS connectatel_users (
      user_id           INT     NOT NULL COMMENT 'Identificador unico del usuario',
      first_name        STRING  NOT NULL COMMENT 'Nombre del usuario',
      last_name         STRING  NOT NULL COMMENT 'Apellido del usuario',
      age               INT     NOT NULL COMMENT 'Edad en años',
      city              STRING           COMMENT 'Ciudad de residencia (puede ser nula)',
      reg_date          TIMESTAMP NOT NULL COMMENT 'Fecha de registro',
      plan              STRING  NOT NULL COMMENT 'Plan contratado: Basico o Premium',
      churn_date        TIMESTAMP        COMMENT 'Fecha de baja (nula si activo)',
      messages_included INT     NOT NULL COMMENT 'Mensajes incluidos en el plan',
      gb_per_month      INT     NOT NULL COMMENT 'GB de datos incluidos por mes',
      minutes_included  INT     NOT NULL COMMENT 'Minutos incluidos en el plan',
      usd_monthly_pay   DOUBLE  NOT NULL COMMENT 'Pago mensual en USD',
      usd_per_gb        DOUBLE  NOT NULL COMMENT 'Costo por GB adicional en USD',
      usd_per_message   DOUBLE  NOT NULL COMMENT 'Costo por mensaje adicional en USD',
      usd_per_minute    DOUBLE  NOT NULL COMMENT 'Costo por minuto adicional en USD'
  )
  USING DELTA
  COMMENT 'Tabla de usuarios de la empresa de telecomunicaciones Connectatel';

##  Seccion 2 — Configuracion de Infraestructura en Databricks CE

  El entorno de ejecucion utilizado es **Databricks Community Edition** con computo **Serverless**,
  el cual aprovisiona recursos automaticamente sin necesidad de configurar un cluster manualmente.

  | Parametro | Valor |
  |---|---|
  | Plataforma | Databricks Community Edition |
  | Tipo de computo | Serverless |
  | Gestion de recursos | Automatica (autoscaling gestionado por Databricks) |
  | Almacenamiento | DBFS (Databricks File System) — `/FileStore/tables/` |
  | Formato de tabla | Delta Lake |

  ![Runtime y configuracipon del cluster - Serverless.png](./Runtime y configuracipon del cluster - Serverless.png "Runtime y configuracipon del cluster - Serverless.png")

In [0]:
import sys

print("=" * 55)
print("       VERSIONES DEL ENTORNO DATABRICKS")
print("=" * 55)
print(f"  Spark version   : {spark.version}")
print(f"  Python version  : {sys.version}")
print("=" * 55)


       VERSIONES DEL ENTORNO DATABRICKS
  Spark version   : 4.1.0
  Python version  : 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]


In [0]:
print("=" * 55)
print("     CONFIGURACION DEL ENTORNO SPARK")
print("=" * 55)

def safe_conf(key):
    try:
        return spark.conf.get(key)
    except Exception:
        return "No disponible en Serverless"

configs = {
    "spark.sql.shuffle.partitions"                      : safe_conf("spark.sql.shuffle.partitions"),
    "spark.sql.adaptive.enabled"                        : safe_conf("spark.sql.adaptive.enabled"),
    "spark.sql.adaptive.coalescePartitions.enabled"     : safe_conf("spark.sql.adaptive.coalescePartitions.enabled"),
    "spark.sql.execution.arrow.pyspark.enabled"         : safe_conf("spark.sql.execution.arrow.pyspark.enabled"),
    "spark.sql.warehouse.id"                            : safe_conf("spark.databricks.warehouse.id"),
}

for key, value in configs.items():
    print(f"  {key}")
    print(f"    -> {value}")
    print()

print("=" * 55)
print("  Configuraciones globales via SQL:")
print("=" * 55)

# Muestra todas las configuraciones disponibles via SQL (compatible Serverless)
display(spark.sql("SET -v"))

     CONFIGURACION DEL ENTORNO SPARK
  spark.sql.shuffle.partitions
    -> auto

  spark.sql.adaptive.enabled
    -> No disponible en Serverless

  spark.sql.adaptive.coalescePartitions.enabled
    -> No disponible en Serverless

  spark.sql.execution.arrow.pyspark.enabled
    -> No disponible en Serverless

  spark.sql.warehouse.id
    -> No disponible en Serverless

  Configuraciones globales via SQL:


key,value,meaning,Since version
spark.databricks.execution.timeout,9000,Timeout in seconds for query executions that can be changed by user.,
spark.sql.ansi.enabled,true,"When true, Spark SQL uses an ANSI compliant dialect instead of being Hive compliant. For example, Spark will throw an exception at runtime instead of returning null results when the inputs to a SQL operator/function are invalid. For full details of this dialect, you can find them in the section ""ANSI Compliance"" of Spark's documentation. Some ANSI dialect features may be not from the ANSI SQL standard directly, but their behaviors align with ANSI SQL's style",3.0.0
spark.sql.files.maxPartitionBytes,128MB,"The maximum number of bytes to pack into a single partition when reading files. This configuration is effective only when using file-based sources such as Parquet, JSON and ORC.",2.0.0
spark.sql.session.timeZone,Etc/UTC,"The ID of session local timezone in the format of either region-based zone IDs or zone offsets. Region IDs must have the form 'area/city', such as 'America/Los_Angeles'. Zone offsets must be in the format '(+|-)HH', '(+|-)HH:mm' or '(+|-)HH:mm:ss', e.g '-08', '+01:00' or '-13:33:33'. Also 'UTC' and 'Z' are supported as aliases of '+00:00'. Other short names are not recommended to use because they can be ambiguous.",2.2.0
spark.sql.shuffle.partitions,auto,"The default number of partitions to use when shuffling data for joins or aggregations. Or set to 'auto' to enable Auto Optimized Shuffle, which will automatically determine this number based on the query plan and the query input data size.",1.1.0


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS main.default;
CREATE VOLUME IF NOT EXISTS main.default.actividad3;

## Creación del volumen requerido
![Volumen creado correctamente.png](./Volumen creado correctamente.png "Volumen creado correctamente.png")

## Detalles del volumen implementado
![Detalles del volumen.png](./Detalles del volumen.png "Detalles del volumen.png")

In [0]:
display(dbutils.fs.ls("/Volumes/main/default/actividad3/"))

[]

In [0]:
print("=" * 55)
print("     ESTRUCTURA DE ALMACENAMIENTO — VOLUME")
print("=" * 55)

items = dbutils.fs.ls("/Volumes/workspace/default/actividad3/")

for item in items:
    size_kb = round(item.size / 1024, 2) if item.size > 0 else 0
    print(f"  Nombre : {item.name}")
    print(f"  Ruta   : {item.path}")
    print(f"  Tamaño : {size_kb} KB")
    print()

print("  Tipo de almacenamiento : Unity Catalog Volume")
print("  Catalogo               : workspace")
print("  Schema                 : default")
print("  Volume                 : actividad3")
print("=" * 55)

     ESTRUCTURA DE ALMACENAMIENTO — VOLUME
  Nombre : users_connectatel.csv
  Ruta   : dbfs:/Volumes/workspace/default/actividad3/users_connectatel.csv
  Tamaño : 367.32 KB

  Tipo de almacenamiento : Unity Catalog Volume
  Catalogo               : workspace
  Schema                 : default
  Volume                 : actividad3


## Seccion 3 — Ingesta de Datos y Creacion de Tabla

  ### Justificacion del metodo de carga

  Se utiliza la **Opcion B (carga manual via UI)** por ser la mas practica en Databricks CE:

  - No requiere token de API de Kaggle ni configuracion adicional
  - El archivo se cargo directamente al **Unity Catalog Volume** `workspace.default.actividad3`
  - Ruta final del archivo: `/Volumes/workspace/default/actividad3/users_connectatel.csv`
  - El esquema disenado en la Seccion 1 sirve como referencia de tipos y nulabilidad

In [0]:
FILE_PATH = "/Volumes/workspace/default/actividad3/users_connectatel.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("nullValue", "?") \
    .option("emptyValue", None) \
    .csv(FILE_PATH)

print("Archivo leido correctamente desde:")
print(f"  {FILE_PATH}\n")
print(f"Total de filas    : {df.count():,}")
print(f"Total de columnas : {len(df.columns)}")
print(f"\nColumnas: {df.columns}")


Archivo leido correctamente desde:
  /Volumes/workspace/default/actividad3/users_connectatel.csv

Total de filas    : 4,000
Total de columnas : 15

Columnas: ['user_id', 'first_name', 'last_name', 'age', 'city', 'reg_date', 'plan', 'churn_date', 'messages_included', 'gb_per_month', 'minutes_included', 'usd_monthly_pay', 'usd_per_gb', 'usd_per_message', 'usd_per_minute']


In [0]:
print("Muestra de los primeros 5 registros:")
df.show(5, truncate=False)

Muestra de los primeros 5 registros:
+-------+----------+---------+---+--------+--------------------------+-------+----------+-----------------+------------+----------------+---------------+----------+---------------+--------------+
|user_id|first_name|last_name|age|city    |reg_date                  |plan   |churn_date|messages_included|gb_per_month|minutes_included|usd_monthly_pay|usd_per_gb|usd_per_message|usd_per_minute|
+-------+----------+---------+---+--------+--------------------------+-------+----------+-----------------+------------+----------------+---------------+----------+---------------+--------------+
|10000  |Carlos    |Garcia   |38 |Medellín|2022-01-01 00:00:00       |Basico |NULL      |100              |5           |100             |12             |1.2       |0.08           |0.1           |
|10001  |Mateo     |Torres   |53 |NULL    |2022-01-01 06:34:17.914478|Basico |NULL      |100              |5           |100             |12             |1.2       |0.08           

In [0]:
TABLE_NAME = "workspace.default.connectatel_users"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_NAME)

print(f"Tabla creada exitosamente: {TABLE_NAME}")

Tabla creada exitosamente: workspace.default.connectatel_users


In [0]:
%sql
DESCRIBE TABLE workspace.default.connectatel_users;

col_name,data_type,comment
user_id,int,null
first_name,string,null
last_name,string,null
age,int,null
city,string,null
reg_date,timestamp,null
plan,string,null
churn_date,string,null
messages_included,int,null
gb_per_month,int,null


## Seccion 4 — Validaciones en Spark y SQL

  Cada validacion se ejecuta en dos formas equivalentes: **PySpark (DataFrame API)** y **Spark SQL**,
  permitiendo comparar resultados y comprender las diferencias de sintaxis entre ambos enfoques.

BLOQUE 1 — Metadatos del esquema

In [0]:
print("=" * 55)
print("  [PySpark] Esquema del DataFrame")
print("=" * 55)
df.printSchema()

  [PySpark] Esquema del DataFrame
root
 |-- user_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- reg_date: timestamp (nullable = true)
 |-- plan: string (nullable = true)
 |-- churn_date: string (nullable = true)
 |-- messages_included: integer (nullable = true)
 |-- gb_per_month: integer (nullable = true)
 |-- minutes_included: integer (nullable = true)
 |-- usd_monthly_pay: integer (nullable = true)
 |-- usd_per_gb: double (nullable = true)
 |-- usd_per_message: double (nullable = true)
 |-- usd_per_minute: double (nullable = true)



In [0]:
%sql
-- [SQL] Metadatos y definicion de la tabla
  DESCRIBE TABLE EXTENDED workspace.default.connectatel_users;

col_name,data_type,comment
user_id,int,null
first_name,string,null
last_name,string,null
age,int,null
city,string,null
reg_date,timestamp,null
plan,string,null
churn_date,string,null
messages_included,int,null
gb_per_month,int,null


BLOQUE 2 — Descripcion estadistica de los datos

In [0]:
print("=" * 55)
print("  [PySpark] Estadisticas descriptivas")
print("=" * 55)
df.describe(
    "age", "usd_monthly_pay", "gb_per_month",
    "minutes_included", "messages_included"
).show(truncate=False)

  [PySpark] Estadisticas descriptivas
+-------+------------------+----------------+-----------------+------------------+------------------+
|summary|age               |usd_monthly_pay |gb_per_month     |minutes_included  |messages_included |
+-------+------------------+----------------+-----------------+------------------+------------------+
|count  |4000              |4000            |4000             |4000              |4000              |
|mean   |33.73975          |16.56625        |10.26875         |275.625           |240.5             |
|stddev |123.23225660193143|6.20646767475243|7.161308855483899|238.71029518279383|190.96823614623506|
|min    |-999              |12              |5                |100               |100               |
|max    |79                |25              |20               |600               |500               |
+-------+------------------+----------------+-----------------+------------------+------------------+



In [0]:
 %sql
  -- [SQL] Estadisticas equivalentes con funciones agregadas
  SELECT
      COUNT(*)                        AS total_registros,
      ROUND(MIN(age), 2)              AS edad_min,
      ROUND(MAX(age), 2)              AS edad_max,
      ROUND(AVG(age), 2)              AS edad_promedio,
      ROUND(MIN(usd_monthly_pay), 2)  AS pago_min_usd,
      ROUND(MAX(usd_monthly_pay), 2)  AS pago_max_usd,
      ROUND(AVG(usd_monthly_pay), 2)  AS pago_promedio_usd
  FROM workspace.default.connectatel_users;

total_registros,edad_min,edad_max,edad_promedio,pago_min_usd,pago_max_usd,pago_promedio_usd
4000,-999,79,33.74,12,25,16.57


BLOQUE 3 — SELECT con filtro

In [0]:
from pyspark.sql.functions import col

print("=" * 55)
print("  [PySpark] Usuarios Premium mayores de 50 años")
print("=" * 55)
df.filter((col("plan") == "Premium") & (col("age") > 50)) \
  .select("user_id", "first_name", "last_name", "age", "city", "plan") \
  .show(10, truncate=False)

  [PySpark] Usuarios Premium mayores de 50 años
+-------+----------+---------+---+--------+-------+
|user_id|first_name|last_name|age|city    |plan   |
+-------+----------+---------+---+--------+-------+
|10003  |Mateo     |Ramirez  |69 |Bogotá  |Premium|
|10007  |Sofia     |Gomez    |70 |Medellín|Premium|
|10011  |Sofia     |Ramirez  |60 |Bogotá  |Premium|
|10013  |Mateo     |Ramirez  |78 |MTY     |Premium|
|10033  |Carlos    |Gomez    |57 |GDL     |Premium|
|10037  |Sofia     |Torres   |79 |Medellín|Premium|
|10039  |Carlos    |Torres   |67 |Bogotá  |Premium|
|10047  |Carlos    |Ramirez  |54 |Bogotá  |Premium|
|10059  |Carlos    |Gomez    |59 |NULL    |Premium|
|10077  |Luis      |Torres   |70 |CDMX    |Premium|
+-------+----------+---------+---+--------+-------+
only showing top 10 rows


In [0]:
%sql
  -- [SQL] Equivalente: usuarios Premium mayores de 50 años
  SELECT user_id, first_name, last_name, age, city, plan
  FROM workspace.default.connectatel_users
  WHERE plan = 'Premium' AND age > 50
  LIMIT 10;

user_id,first_name,last_name,age,city,plan
10003,Mateo,Ramirez,69,Bogotá,Premium
10007,Sofia,Gomez,70,Medellín,Premium
10011,Sofia,Ramirez,60,Bogotá,Premium
10013,Mateo,Ramirez,78,MTY,Premium
10033,Carlos,Gomez,57,GDL,Premium
10037,Sofia,Torres,79,Medellín,Premium
10039,Carlos,Torres,67,Bogotá,Premium
10047,Carlos,Ramirez,54,Bogotá,Premium
10059,Carlos,Gomez,59,null,Premium
10077,Luis,Torres,70,CDMX,Premium


BLOQUE 4 — GROUP BY por plan

In [0]:
from pyspark.sql.functions import count, avg, round as spark_round, sum as spark_sum

print("=" * 55)
print("  [PySpark] Conteo y metricas agrupadas por plan")
print("=" * 55)
df.groupBy("plan") \
   .agg(
       count("*").alias("total_usuarios"),
       spark_round(avg("age"), 2).alias("edad_promedio"),
       spark_round(avg("usd_monthly_pay"), 2).alias("pago_promedio_usd"),
       spark_round(avg("gb_per_month"), 2).alias("gb_promedio")
   ) \
   .orderBy("plan") \
   .show(truncate=False)

  [PySpark] Conteo y metricas agrupadas por plan
+-------+--------------+-------------+-----------------+-----------+
|plan   |total_usuarios|edad_promedio|pago_promedio_usd|gb_promedio|
+-------+--------------+-------------+-----------------+-----------+
|Basico |2595          |31.5         |12.0             |5.0        |
|Premium|1405          |37.88        |25.0             |20.0       |
+-------+--------------+-------------+-----------------+-----------+



In [0]:
%sql
  -- [SQL] Equivalente: metricas agrupadas por plan
  SELECT
      plan,
      COUNT(*)                        AS total_usuarios,
      ROUND(AVG(age), 2)              AS edad_promedio,
      ROUND(AVG(usd_monthly_pay), 2)  AS pago_promedio_usd,
      ROUND(AVG(gb_per_month), 2)     AS gb_promedio
  FROM workspace.default.connectatel_users
  GROUP BY plan
  ORDER BY plan;

plan,total_usuarios,edad_promedio,pago_promedio_usd,gb_promedio
Basico,2595,31.5,12.0,5.0
Premium,1405,37.88,25.0,20.0


BLOQUE 5 — GROUP BY por ciudad

In [0]:
print("=" * 55)
print("  [PySpark] Top 5 ciudades con mas usuarios")
print("=" * 55)
df.filter(col("city").isNotNull()) \
  .groupBy("city") \
  .agg(count("*").alias("total_usuarios")) \
  .orderBy(col("total_usuarios").desc()) \
  .show(5, truncate=False)

  [PySpark] Top 5 ciudades con mas usuarios
+--------+--------------+
|city    |total_usuarios|
+--------+--------------+
|Bogotá  |808           |
|CDMX    |730           |
|Medellín|616           |
|GDL     |450           |
|Cali    |424           |
+--------+--------------+
only showing top 5 rows


In [0]:
%sql
  -- [SQL] Equivalente: top 5 ciudades con mas usuarios
  SELECT
      city,
      COUNT(*) AS total_usuarios
  FROM workspace.default.connectatel_users
  WHERE city IS NOT NULL
  GROUP BY city
  ORDER BY total_usuarios DESC
  LIMIT 5;

city,total_usuarios
Bogotá,808
CDMX,730
Medellín,616
GDL,450
Cali,424


BLOQUE 6 — Conteo de nulos y usuarios activos vs inactivos

In [0]:
from pyspark.sql.functions import when

print("=" * 55)
print("  [PySpark] Usuarios activos vs dados de baja (churn)")
print("=" * 55)
df.groupBy(
    when(col("churn_date").isNull(), "Activo")
    .otherwise("Dado de baja").alias("estado")
) \
  .agg(count("*").alias("total")) \
  .orderBy("estado") \
  .show(truncate=False)

  [PySpark] Usuarios activos vs dados de baja (churn)
+------------+-----+
|estado      |total|
+------------+-----+
|Activo      |3534 |
|Dado de baja|466  |
+------------+-----+



In [0]:
%sql
  -- [SQL] Equivalente: usuarios activos vs dados de baja
  SELECT
      CASE WHEN churn_date IS NULL THEN 'Activo'
           ELSE 'Dado de baja'
      END AS estado,
      COUNT(*) AS total
  FROM workspace.default.connectatel_users
  GROUP BY
      CASE WHEN churn_date IS NULL THEN 'Activo'
           ELSE 'Dado de baja'
      END
  ORDER BY estado;

estado,total
Activo,3534
Dado de baja,466


## Seccion 5 — Ventajas y Desventajas: SQL vs Spark (PySpark)

###   Tabla Comparativa

  | # | Criterio | SQL | Spark (PySpark) |
  |---|---|---|---|
  | 1 | **Facilidad de uso** | Alta — sintaxis declarativa familiar para cualquier analista o profesional de datos sin experiencia en programacion | Media/Alta — requiere conocimiento de Python y la API de DataFrames; curva de aprendizaje mas pronunciada |
  | 2 | **Expresividad** | Limitada para logica compleja — dificil expresar iteraciones, condiciones anidadas o transformaciones personalizadas | Alta — permite logica arbitraria con Python, UDFs, lambdas y control de flujo completo |
  | 3 | **Escalabilidad** | Depende del motor subyacente; en motores tradicionales escala con dificultad ante volumenes masivos | Nativa — disenado para procesar datos distribuidos a escala de petabytes en clusters de multiples nodos |
  | 4 | **Integracion con herramientas BI** | Excelente — herramientas como Power BI, Tableau y Looker se conectan directamente via JDBC/ODBC | Limitada — generalmente requiere exponer los datos como tabla SQL o exportar antes de visualizar |
  | 5 | **Pipelines y ML** | Inadecuado para pipelines complejos de ingenieria de datos o entrenamiento de modelos | Ideal — integra nativamente con MLlib, SparkML y frameworks externos como scikit-learn via pandas UDFs |

### Conclusion

  Ambos paradigmas son complementarios dentro del ecosistema Databricks:

  - **SQL** es la eleccion natural para exploracion rapida, reportes y consultas ad-hoc comprensibles para equipos
  multidisciplinarios.
  - **PySpark** es indispensable cuando se requiere escalabilidad real, transformaciones complejas, pipelines de ETL
  robustos o integracion con modelos de machine learning.

  En la practica, la estrategia mas efectiva es combinar ambos: usar **SQL para validar y explorar**, y **PySpark para
  transformar y escalar**.

  ## Conclusiones Generales de la Actividad

  | Fase | Resultado |
  |---|---|
  | Diseño del esquema | StructType definido con 15 campos, tipos variados y nulabilidad documentada |
  | Infraestructura | Databricks CE con Serverless compute — configuracion verificada |
  | Ingesta | Dataset de 4.000 registros cargado desde Unity Catalog Volume a tabla Delta |
  | Validaciones | 6 bloques de validacion ejecutados en PySpark y SQL con resultados equivalentes |
  | Analisis comparativo | SQL y PySpark evaluados en 5 criterios clave con conclusion de uso complementario |

  **Dataset utilizado:** `users_connectatel.csv` — Empresa de telecomunicaciones Connectatel
  **Almacenamiento:** Unity Catalog Volume `workspace.default.actividad3` — Formato Delta Lake
  **Tabla final:** `workspace.default.connectatel_users`